# 03 - Full MultiTask Emotion2Vec CoAttention 5-Fold

Notebook này train **full proposed model**, không còn là metadata baseline.

Input bắt buộc:

```text
data/features/iemocap_full_emotion2vec_acoustic_cache.npz
```

Kiến trúc:

```text
Branch A: Emotion2Vec frozen embedding -> adapter MLP -> z_e
Branch B: handcrafted acoustic vector -> acoustic MLP -> z_a
Cross-attention: z_a queries z_e, z_e queries z_a
Gated fusion -> shared representation
Head 1: emotion classification
Head 2: valence/arousal/dominance regression
```

In [ ]:
from pathlib import Path
import json
import math
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

SEED = int(os.getenv("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "Speech Project" and PROJECT_ROOT.parent.name == "Speech Project":
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

BASE_DIR = PROJECT_ROOT / "06_w2v_based_models"
if not BASE_DIR.exists() and Path("/kaggle/working").exists():
    BASE_DIR = Path("/kaggle/working") / "06_w2v_based_models"

LOCAL_DATA_DIR = PROJECT_ROOT / "06_w2v_based_models" / "data"

def has_train_tables(path):
    return (
        path is not None
        and (path / "metadata" / "iemocap_metadata_full.csv").exists()
        and (path / "splits").exists()
    )

def candidate_data_roots(base):
    if base is None:
        return []
    base = Path(base)
    roots = [base, base / "data"]
    if base.exists():
        for meta_path in base.rglob("iemocap_metadata_full.csv"):
            roots.append(meta_path.parent.parent)
    return roots

def resolve_data_dir():
    candidates = []
    env_dir = os.environ.get("IEMOCAP_DATA_DIR")
    if env_dir:
        candidates.append(Path(env_dir))
    candidates.extend([
        LOCAL_DATA_DIR,
        Path("/kaggle/input/iemocap-full-multitask-data"),
        Path("/kaggle/input/iemocap_full_multitask_data"),
        Path("/kaggle/input/iemocap-multitask-train-data"),
        Path("/kaggle/input/iemocap_multitask_train_data"),
    ])

    for candidate in candidates:
        for root in candidate_data_roots(candidate):
            if has_train_tables(root):
                return root.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for root in candidate_data_roots(kaggle_input):
            if has_train_tables(root):
                return root.resolve()

    raise FileNotFoundError(
        "Không tìm thấy dữ liệu IEMOCAP. Notebook sẽ tự quét /kaggle/input, nhưng folder dữ liệu cần có "
        "metadata/iemocap_metadata_full.csv và splits/. Nếu bạn đặt ở chỗ khác, set IEMOCAP_DATA_DIR."
    )

DATA_DIR = resolve_data_dir()
METADATA_DIR = DATA_DIR / "metadata"
SPLIT_DIR = DATA_DIR / "splits"
AUDIO_DIR = DATA_DIR / "audio_wav"

FULL_METADATA_PATH = METADATA_DIR / "iemocap_metadata_full.csv"

WORKING_DATA_DIR = Path("/kaggle/working/iemocap_full_multitask_data") if Path("/kaggle/working").exists() else DATA_DIR
WORKING_FEATURE_DIR = WORKING_DATA_DIR / "features"
WORKING_REPORT_DIR = WORKING_DATA_DIR / "feature_reports"
WORKING_FIGURE_DIR = WORKING_DATA_DIR / "feature_figures"
INPUT_FEATURE_DIR = DATA_DIR / "features"
FEATURE_CACHE_NAME = "iemocap_full_emotion2vec_acoustic_cache.npz"
WORKING_FEATURE_CACHE_PATH = WORKING_FEATURE_DIR / FEATURE_CACHE_NAME
INPUT_FEATURE_CACHE_PATH = INPUT_FEATURE_DIR / FEATURE_CACHE_NAME

def resolve_feature_cache_path(require_exists=False):
    env_cache = os.environ.get("IEMOCAP_FEATURE_CACHE", "").strip()
    candidates = []
    if env_cache:
        candidates.append(Path(env_cache))
    candidates.extend([WORKING_FEATURE_CACHE_PATH, INPUT_FEATURE_CACHE_PATH])
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    if require_exists:
        raise FileNotFoundError(
            "Không tìm thấy full feature cache. Hãy chạy notebook 02 trước. Trên Kaggle, notebook 02 ghi cache vào "
            f"{WORKING_FEATURE_CACHE_PATH}, không ghi vào /kaggle/input vì Kaggle input dataset là read-only."
        )
    return WORKING_FEATURE_CACHE_PATH

print("DATA_DIR:", DATA_DIR)
print("AUDIO_DIR:", AUDIO_DIR, AUDIO_DIR.exists())
print("INPUT_FEATURE_CACHE_PATH:", INPUT_FEATURE_CACHE_PATH, INPUT_FEATURE_CACHE_PATH.exists())
print("WORKING_FEATURE_CACHE_PATH:", WORKING_FEATURE_CACHE_PATH, WORKING_FEATURE_CACHE_PATH.exists())
print("WORKING_REPORT_DIR:", WORKING_REPORT_DIR)
print("WORKING_FIGURE_DIR:", WORKING_FIGURE_DIR)

In [ ]:
NOTEBOOK_DIR = BASE_DIR / "03MultiTask Emotion2Vec CoAttention 5Fold"
REPORT_DIR = NOTEBOOK_DIR / "reports"
FIGURE_DIR = NOTEBOOK_DIR / "figures"
PREDICTION_DIR = NOTEBOOK_DIR / "predictions"
MODEL_DIR = NOTEBOOK_DIR / "models"
for folder in [REPORT_DIR, FIGURE_DIR, PREDICTION_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PROTOCOL = "5fold_session"
SPLIT_PATH = SPLIT_DIR / "iemocap_5fold_session_long.csv"
N_EXPECTED_FOLDS = 5
FEATURE_CACHE_PATH = resolve_feature_cache_path(require_exists=True)
print("SPLIT_PATH:", SPLIT_PATH, SPLIT_PATH.exists())
print("FEATURE_CACHE_PATH:", FEATURE_CACHE_PATH, FEATURE_CACHE_PATH.exists())

In [ ]:
EMOTIONS = ["neutral", "angry", "sad", "happy"]
ID_TO_EMOTION = {0: "neutral", 1: "angry", 2: "sad", 3: "happy"}

EPOCHS = int(os.getenv("EPOCHS", "25"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "64"))
LR = float(os.getenv("LR", "8e-4"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "1e-4"))
PATIENCE = int(os.getenv("PATIENCE", "7"))
LOSS_EMOTION_WEIGHT = float(os.getenv("LOSS_EMOTION_WEIGHT", "1.0"))
LOSS_AVD_WEIGHT = float(os.getenv("LOSS_AVD_WEIGHT", "0.35"))
LOSS_CCC_WEIGHT = float(os.getenv("LOSS_CCC_WEIGHT", "0.65"))

def normalize_avd(df):
    out = df.copy()
    for src, dst in [("valence", "valence_norm"), ("arousal", "arousal_norm"), ("dominance", "dominance_norm")]:
        if dst not in out.columns:
            out[dst] = (pd.to_numeric(out[src], errors="coerce") - 1.0) / 4.0
        out[dst] = out[dst].clip(0.0, 1.0)
    return out

def load_feature_cache(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy full feature cache: {path}. Hãy chạy notebook 02 trước, hoặc upload file .npz này vào data/features/."
        )
    cache = np.load(path, allow_pickle=True)
    return {
        "sample_ids": cache["sample_ids"].astype(str),
        "emotion2vec": cache["emotion2vec"].astype(np.float32),
        "acoustic": cache["acoustic"].astype(np.float32),
        "acoustic_names": cache["acoustic_names"].astype(str) if "acoustic_names" in cache else np.array([]),
    }

def load_and_align_split(split_path, metadata_path, cache):
    split_df = pd.read_csv(split_path)
    meta = pd.read_csv(metadata_path)
    extra_cols = ["train_sample_id", "transcription", "source_filename", "duration", "sample_rate", "channels"]
    extra_cols = [c for c in extra_cols if c in meta.columns]
    split_df = split_df.merge(meta[extra_cols].drop_duplicates("train_sample_id"), on="train_sample_id", how="left")
    split_df = normalize_avd(split_df)

    id_to_idx = {sid: i for i, sid in enumerate(cache["sample_ids"])}
    split_df["feature_idx"] = split_df["train_sample_id"].astype(str).map(id_to_idx)
    missing = split_df["feature_idx"].isna().sum()
    if missing:
        missing_ids = split_df.loc[split_df["feature_idx"].isna(), "train_sample_id"].head(10).tolist()
        raise ValueError(f"{missing} rows have no feature cache. First missing ids: {missing_ids}")
    split_df["feature_idx"] = split_df["feature_idx"].astype(int)
    return split_df

class FullFeatureDataset(Dataset):
    def __init__(self, df, cache, e_scaler=None, a_scaler=None, fit_scalers=False):
        self.df = df.reset_index(drop=True).copy()
        idx = self.df["feature_idx"].values
        e = cache["emotion2vec"][idx]
        a = cache["acoustic"][idx]
        if fit_scalers:
            self.e_scaler = StandardScaler().fit(e)
            self.a_scaler = StandardScaler().fit(a)
        else:
            self.e_scaler = e_scaler
            self.a_scaler = a_scaler
        self.e = torch.tensor(self.e_scaler.transform(e), dtype=torch.float32)
        self.a = torch.tensor(self.a_scaler.transform(a), dtype=torch.float32)
        self.y_cls = torch.tensor(self.df["emotion_id"].values, dtype=torch.long)
        self.y_reg = torch.tensor(self.df[["valence_norm", "arousal_norm", "dominance_norm"]].values, dtype=torch.float32)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        return {"e2v": self.e[idx], "acoustic": self.a[idx], "emotion": self.y_cls[idx], "avd": self.y_reg[idx], "idx": idx}

class MLPBranch(nn.Module):
    def __init__(self, in_dim, out_dim=192, hidden=384, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
        )
    def forward(self, x):
        return self.net(x)

class Emotion2VecAcousticCoAttentionMultiTask(nn.Module):
    """
    Full proposed model:
      Branch A: frozen emotion2vec embedding -> adapter MLP -> z_e
      Branch B: handcrafted acoustic vector -> acoustic MLP -> z_a
      Cross attention: acoustic query attends to emotion2vec and emotion2vec query attends to acoustic
      Fusion: gated fusion -> shared representation
      Head 1: 4-class emotion classification
      Head 2: valence/arousal/dominance regression
    """
    def __init__(self, e2v_dim, acoustic_dim, hidden=192, heads=4, dropout=0.25, n_classes=4):
        super().__init__()
        self.e2v_branch = MLPBranch(e2v_dim, out_dim=hidden, hidden=hidden * 2, dropout=dropout)
        self.acoustic_branch = MLPBranch(acoustic_dim, out_dim=hidden, hidden=hidden * 2, dropout=dropout)
        self.a_queries_e = nn.MultiheadAttention(hidden, heads, dropout=0.1, batch_first=True)
        self.e_queries_a = nn.MultiheadAttention(hidden, heads, dropout=0.1, batch_first=True)
        self.gate = nn.Sequential(nn.Linear(hidden * 4, hidden), nn.Sigmoid())
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 4, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.emotion_head = nn.Linear(hidden, n_classes)
        self.avd_head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden // 2, 3), nn.Sigmoid())
    def forward(self, e2v, acoustic):
        z_e = self.e2v_branch(e2v)
        z_a = self.acoustic_branch(acoustic)
        e_tok = z_e.unsqueeze(1)
        a_tok = z_a.unsqueeze(1)
        a_ctx, _ = self.a_queries_e(query=a_tok, key=e_tok, value=e_tok)
        e_ctx, _ = self.e_queries_a(query=e_tok, key=a_tok, value=a_tok)
        parts = torch.cat([z_e, z_a, e_ctx.squeeze(1), a_ctx.squeeze(1)], dim=-1)
        g = self.gate(parts)
        fused = self.fusion(parts)
        z = g * fused + (1.0 - g) * 0.5 * (z_e + z_a)
        return self.emotion_head(z), self.avd_head(z)

def ccc_torch(pred, target, eps=1e-8):
    pred_mean = torch.mean(pred, dim=0)
    target_mean = torch.mean(target, dim=0)
    pred_var = torch.var(pred, dim=0, unbiased=False)
    target_var = torch.var(target, dim=0, unbiased=False)
    cov = torch.mean((pred - pred_mean) * (target - target_mean), dim=0)
    return (2.0 * cov) / (pred_var + target_var + (pred_mean - target_mean) ** 2 + eps)

def ccc_loss(pred, target):
    return 1.0 - ccc_torch(pred, target).mean()

def ccc_np(pred, target, eps=1e-8):
    vals = []
    for i in range(pred.shape[1]):
        x, y = pred[:, i], target[:, i]
        mx, my = x.mean(), y.mean()
        vx, vy = x.var(), y.var()
        cov = np.mean((x - mx) * (y - my))
        vals.append((2 * cov) / (vx + vy + (mx - my) ** 2 + eps))
    return np.asarray(vals)

def class_weights(train_df):
    counts = train_df["emotion_id"].value_counts().sort_index()
    total = counts.sum()
    return torch.tensor([total / (4.0 * counts.get(i, 1)) for i in range(4)], dtype=torch.float32)

def run_epoch(model, loader, optimizer, ce):
    model.train()
    losses = []
    for batch in loader:
        e = batch["e2v"].to(DEVICE)
        a = batch["acoustic"].to(DEVICE)
        yc = batch["emotion"].to(DEVICE)
        yr = batch["avd"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits, pred = model(e, a)
        loss = LOSS_EMOTION_WEIGHT * ce(logits, yc) + LOSS_AVD_WEIGHT * nn.functional.smooth_l1_loss(pred, yr) + LOSS_CCC_WEIGHT * ccc_loss(pred, yr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

@torch.no_grad()
def predict(model, loader):
    model.eval()
    logits, reg, yc, yr = [], [], [], []
    for batch in loader:
        lo, pr = model(batch["e2v"].to(DEVICE), batch["acoustic"].to(DEVICE))
        logits.append(lo.cpu().numpy())
        reg.append(pr.cpu().numpy())
        yc.append(batch["emotion"].numpy())
        yr.append(batch["avd"].numpy())
    return {"logits": np.concatenate(logits), "reg": np.concatenate(reg), "y_cls": np.concatenate(yc), "y_reg": np.concatenate(yr)}

def compute_metrics(pred):
    y_true = pred["y_cls"]
    y_pred = pred["logits"].argmax(axis=1)
    ccc = ccc_np(pred["reg"], pred["y_reg"])
    mae = np.mean(np.abs(pred["reg"] - pred["y_reg"]), axis=0)
    rmse = np.sqrt(np.mean((pred["reg"] - pred["y_reg"]) ** 2, axis=0))
    return {
        "WA": accuracy_score(y_true, y_pred),
        "UAR": balanced_accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted_F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "CCC_valence": ccc[0],
        "CCC_arousal": ccc[1],
        "CCC_dominance": ccc[2],
        "CCC_mean": float(ccc.mean()),
        "MAE_mean": float(mae.mean()),
        "RMSE_mean": float(rmse.mean()),
    }

def primary_score(m):
    return float(m["UAR"] + m["CCC_mean"])

In [ ]:
cache = load_feature_cache(FEATURE_CACHE_PATH)
split_df = load_and_align_split(SPLIT_PATH, FULL_METADATA_PATH, cache)
print("Split rows:", split_df.shape)
print("Feature shapes:", cache["emotion2vec"].shape, cache["acoustic"].shape)
print("Folds:", split_df["fold"].nunique())
if split_df["fold"].nunique() != N_EXPECTED_FOLDS:
    raise ValueError(f"Expected {N_EXPECTED_FOLDS} folds")
display(split_df.head())

In [ ]:
def run_fold(fold):
    train_df = split_df[(split_df["fold"].eq(fold)) & (split_df["split"].eq("train"))].copy()
    val_df = split_df[(split_df["fold"].eq(fold)) & (split_df["split"].eq("validation"))].copy()
    test_df = split_df[(split_df["fold"].eq(fold)) & (split_df["split"].eq("test"))].copy()

    train_ds = FullFeatureDataset(train_df, cache, fit_scalers=True)
    val_ds = FullFeatureDataset(val_df, cache, e_scaler=train_ds.e_scaler, a_scaler=train_ds.a_scaler)
    test_ds = FullFeatureDataset(test_df, cache, e_scaler=train_ds.e_scaler, a_scaler=train_ds.a_scaler)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = Emotion2VecAcousticCoAttentionMultiTask(
        e2v_dim=cache["emotion2vec"].shape[1],
        acoustic_dim=cache["acoustic"].shape[1],
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    ce = nn.CrossEntropyLoss(weight=class_weights(train_df).to(DEVICE), label_smoothing=0.05)

    best_state = None
    best_score = -1e9
    best_epoch = 0
    patience_left = PATIENCE
    history = []
    for epoch in range(1, EPOCHS + 1):
        loss = run_epoch(model, train_loader, optimizer, ce)
        val_pred = predict(model, val_loader)
        val_metrics = compute_metrics(val_pred)
        score = primary_score(val_metrics)
        history.append({"fold": fold, "epoch": epoch, "train_loss": loss, "primary_score": score, **{f"val_{k}": v for k, v in val_metrics.items()}})
        if score > best_score:
            best_score = score
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
        if patience_left <= 0:
            break

    model.load_state_dict(best_state)
    test_pred = predict(model, test_loader)
    metrics = compute_metrics(test_pred)
    pred_cls = test_pred["logits"].argmax(axis=1)
    probs = torch.softmax(torch.tensor(test_pred["logits"]), dim=1).numpy()

    out = test_df.reset_index(drop=True).copy()
    out["pred_emotion_id"] = pred_cls
    out["pred_emotion"] = [ID_TO_EMOTION[int(x)] for x in pred_cls]
    for i, name in ID_TO_EMOTION.items():
        out[f"prob_{name}"] = probs[:, i]
    out["pred_valence"] = test_pred["reg"][:, 0] * 4.0 + 1.0
    out["pred_arousal"] = test_pred["reg"][:, 1] * 4.0 + 1.0
    out["pred_dominance"] = test_pred["reg"][:, 2] * 4.0 + 1.0

    safe = str(fold).replace(" ", "_").replace("/", "_")
    pd.DataFrame(history).to_csv(REPORT_DIR / f"{safe}_history.csv", index=False, encoding="utf-8-sig")
    out.to_csv(PREDICTION_DIR / f"{safe}_predictions.csv", index=False, encoding="utf-8-sig")
    torch.save({"model_state_dict": model.state_dict(), "fold": fold, "best_epoch": best_epoch, "metrics": metrics}, MODEL_DIR / f"{safe}_best.pt")

    return {"protocol": PROTOCOL, "fold": fold, "best_epoch": best_epoch, "n_train": len(train_df), "n_validation": len(val_df), "n_test": len(test_df), **metrics}, out

rows = []
preds = []
start = time.time()
for fold in sorted(split_df["fold"].unique()):
    print("Running fold:", fold)
    row, pred = run_fold(fold)
    rows.append(row)
    preds.append(pred)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items() if k in ["fold", "WA", "UAR", "Macro_F1", "CCC_mean"]})

metrics_df = pd.DataFrame(rows)
all_predictions = pd.concat(preds, ignore_index=True)
metrics_df.to_csv(REPORT_DIR / f"{PROTOCOL}_full_model_metrics.csv", index=False, encoding="utf-8-sig")
all_predictions.to_csv(PREDICTION_DIR / f"{PROTOCOL}_full_model_predictions.csv", index=False, encoding="utf-8-sig")
print("Seconds:", round(time.time() - start, 2))
display(metrics_df)

In [ ]:
summary = metrics_df.drop(columns=["protocol", "fold"], errors="ignore").select_dtypes(include=[np.number]).agg(["mean", "std"]).T.reset_index()
summary.columns = ["metric", "mean", "std"]
summary.to_csv(REPORT_DIR / f"{PROTOCOL}_full_model_summary.csv", index=False, encoding="utf-8-sig")
display(summary)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    plot_df = metrics_df.melt(id_vars=["fold"], value_vars=["WA", "UAR", "Macro_F1", "CCC_mean"], var_name="metric", value_name="value")
    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x="fold", y="value", hue="metric")
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{PROTOCOL}_full_model_fold_metrics.png", dpi=180)
    plt.show()
except Exception as exc:
    print("Figure skipped:", exc)

report = [
    f"# {PROTOCOL} Full Emotion2Vec + Acoustic Multi-Task Report",
    "",
    "- Feature cache: `data/features/iemocap_full_emotion2vec_acoustic_cache.npz`.",
    "- Branch A: frozen emotion2vec embedding + adapter.",
    "- Branch B: handcrafted acoustic vector + adapter.",
    "- Fusion: bidirectional cross-attention + gated fusion.",
    "- Heads: emotion classification and AVD regression.",
    "",
    "## Mean +/- std",
]
for _, row in summary.iterrows():
    report.append(f"- {row['metric']}: {row['mean']:.4f} +/- {row['std']:.4f}")
report_path = REPORT_DIR / f"{PROTOCOL}_full_model_report.md"
report_path.write_text("\n".join(report), encoding="utf-8")
display(Markdown("\n".join(report)))